In [48]:
import pandas as pd
import numpy as np

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "Hours": [1, 2, 3, 4, 5, 6],
    "Passed": [0, 0, 0, 1, 1, 1]
})

print(df)

    Hours  Passed
0       1       0
1       2       0
2       3       0
3       4       0
4       5       1
5       6       0
6       7       1
7       8       1
8       9       1
9      10       1
10     11       1
11     12       1


In [3]:
bins = pd.cut(df['Hours'],bins=3)

In [4]:
bins

0    (0.995, 2.667]
1    (0.995, 2.667]
2    (2.667, 4.333]
3    (2.667, 4.333]
4      (4.333, 6.0]
5      (4.333, 6.0]
Name: Hours, dtype: category
Categories (3, interval[float64, right]): [(0.995, 2.667] < (2.667, 4.333] < (4.333, 6.0]]

In [8]:
base_prediction = df['Passed'].mean()
base_prediction

np.float64(0.5)

In [11]:
df['Prediction'] = np.log(base_prediction/(1-base_prediction))
df['Raw'] = base_prediction

In [13]:
df['gradient'] = df['Raw']-df['Passed']

In [15]:
df['Hessian'] = df['Raw']*(1-df['Raw'])

In [ ]:
df

In [5]:
df['bins'] = bins

In [19]:
stats = df.groupby('bins', observed=True).agg(
    Count=('Passed', 'count'),
    Passed_sum=('Passed', 'sum'),
    
    Gradient_sum = ('gradient','sum'),
    Hessian_sum = ('Hessian','sum')
)

In [20]:
stats

,Count,Passed_sum,Gradient_sum,Hessian_sum
bins,,,,
"(0.995, 2.667]",2,0,1.0,0.5
"(2.667, 4.333]",2,1,0.0,0.5
"(4.333, 6.0]",2,2,-1.0,0.5


In [21]:
stats['Cumulative_count'] = stats['Count'].cumsum()

In [24]:
stats['Passed_cummulative_sum'] = stats['Passed_sum'].cumsum()

In [25]:
stats['Gradient_cummulative_sum'] = stats['Gradient_sum'].cumsum()

In [26]:
stats['Hessian_cummulative_sum'] = stats['Hessian_sum'].cumsum()

In [27]:
stats

,Count,Passed_sum,Gradient_sum,Hessian_sum,Cumulative_count,Passed_cummulative_sum,Gradient_cummulative_sum,Hessian_cummulative_sum
bins,,,,,,,,
"(0.995, 2.667]",2,2,1.0,0.5,2,2,1.0,0.5
"(2.667, 4.333]",2,4,0.0,0.5,4,6,1.0,1.0
"(4.333, 6.0]",2,6,-1.0,0.5,6,12,0.0,1.5


In [28]:
def gain(g,h,l):
    return (g*g)/(h+l)

In [29]:
last_gradient = stats['Gradient_cummulative_sum'].iloc[-1]
last_hessian = stats['Hessian_cummulative_sum'].iloc[-1]
gains = []
gainp = gain(last_gradient,last_hessian,1)
for i in range(0,len(stats)-1):
    lgrad = stats['Gradient_cummulative_sum'].iloc[i]
    rgrad = last_gradient-lgrad
    lhess = stats['Hessian_cummulative_sum'].iloc[i]
    rhess = last_hessian-lhess
    gains.append(0.5*(gain(lgrad,lhess,1)+gain(rgrad,rhess,1)-gainp))
print(gains)


[np.float64(0.5833333333333333), np.float64(0.5833333333333333)]


In [30]:
best_split = np.argmax(gains)
print(best_split)

0


In [31]:
error = []
last_gradient = stats['Gradient_cummulative_sum'].iloc[-1]
last_hessian = stats['Hessian_cummulative_sum'].iloc[-1]
lgrad = stats['Gradient_cummulative_sum'].iloc[best_split]
rgrad = last_gradient-lgrad
lhess = stats['Hessian_cummulative_sum'].iloc[best_split]
rhess = last_hessian-lhess
error.append(-lgrad/(lhess+1))
error.append(-rgrad/(rhess+1))

In [33]:
error

[np.float64(-0.6666666666666666), np.float64(0.5)]

In [ ]:
print

,Hours,Passed,bins,Prediction,Raw,gradient,Hessian
0,1,0,"(0.995, 2.667]",0.0,0.5,0.5,0.25
1,2,0,"(0.995, 2.667]",0.0,0.5,0.5,0.25
2,3,0,"(2.667, 4.333]",0.0,0.5,0.5,0.25
3,4,1,"(2.667, 4.333]",0.0,0.5,-0.5,0.25
4,5,1,"(4.333, 6.0]",0.0,0.5,-0.5,0.25
5,6,1,"(4.333, 6.0]",0.0,0.5,-0.5,0.25


In [36]:
threshold = stats.index[best_split].right

In [37]:
df['Prediction']+=np.where(df['Hours']<=threshold,0.1*error[0],0.1*error[1])

In [38]:
df

,Hours,Passed,bins,Prediction,Raw,gradient,Hessian
0,1,0,"(0.995, 2.667]",-0.066667,0.5,0.5,0.25
1,2,0,"(0.995, 2.667]",-0.066667,0.5,0.5,0.25
2,3,0,"(2.667, 4.333]",0.050000,0.5,0.5,0.25
3,4,1,"(2.667, 4.333]",0.050000,0.5,-0.5,0.25
4,5,1,"(4.333, 6.0]",0.050000,0.5,-0.5,0.25
5,6,1,"(4.333, 6.0]",0.050000,0.5,-0.5,0.25


In [41]:
df["Raw"] = 1 / (1 + np.exp(-df["Prediction"]))

In [42]:
df

,Hours,Passed,bins,Prediction,Raw,gradient,Hessian
0,1,0,"(0.995, 2.667]",-0.066667,0.483340,0.5,0.25
1,2,0,"(0.995, 2.667]",-0.066667,0.483340,0.5,0.25
2,3,0,"(2.667, 4.333]",0.050000,0.512497,0.5,0.25
3,4,1,"(2.667, 4.333]",0.050000,0.512497,-0.5,0.25
4,5,1,"(4.333, 6.0]",0.050000,0.512497,-0.5,0.25
5,6,1,"(4.333, 6.0]",0.050000,0.512497,-0.5,0.25


In [43]:
import pandas as pd
import numpy as np

def gain(g, h, l):
    return (g * g) / (h + l)

def train_xgboost(df, n_trees, learning_rate=0.1, lambda_reg=1, bins=3):
    df = df.copy()

    base_prediction = df["Passed"].mean()
    base_prediction = np.clip(base_prediction, 1e-10, 1 - 1e-10)

    df["Prediction"] = np.log(base_prediction / (1 - base_prediction))
    df["Raw"] = 1 / (1 + np.exp(-df["Prediction"]))
    df["gradient"] = df["Raw"] - df["Passed"]
    df["Hessian"] = df["Raw"] * (1 - df["Raw"])

    df["bins"] = pd.cut(df["Hours"], bins=bins)

    trees = []

    for _ in range(n_trees):
        stats = df.groupby("bins", observed=True).agg(
            Count=("Passed", "count"),
            Gradient_sum=("gradient", "sum"),
            Hessian_sum=("Hessian", "sum")
        )

        stats["Gradient_cummulative_sum"] = stats["Gradient_sum"].cumsum()
        stats["Hessian_cummulative_sum"] = stats["Hessian_sum"].cumsum()

        last_gradient = stats["Gradient_cummulative_sum"].iloc[-1]
        last_hessian = stats["Hessian_cummulative_sum"].iloc[-1]

        parent_gain = gain(last_gradient, last_hessian, lambda_reg)
        gains = []

        for i in range(len(stats) - 1):
            lgrad = stats["Gradient_cummulative_sum"].iloc[i]
            lhess = stats["Hessian_cummulative_sum"].iloc[i]

            rgrad = last_gradient - lgrad
            rhess = last_hessian - lhess

            split_gain = 0.5 * (
                gain(lgrad, lhess, lambda_reg)
                + gain(rgrad, rhess, lambda_reg)
                - parent_gain
            )

            gains.append(split_gain)

        if not gains:
            break

        best_split = int(np.argmax(gains))
        threshold = stats.index[best_split].right

        lgrad = stats["Gradient_cummulative_sum"].iloc[best_split]
        lhess = stats["Hessian_cummulative_sum"].iloc[best_split]

        rgrad = last_gradient - lgrad
        rhess = last_hessian - lhess

        left_weight = -lgrad / (lhess + lambda_reg)
        right_weight = -rgrad / (rhess + lambda_reg)

        left_mask = df["Hours"] <= threshold

        df.loc[left_mask, "Prediction"] += learning_rate * left_weight
        df.loc[~left_mask, "Prediction"] += learning_rate * right_weight

        df["Raw"] = 1 / (1 + np.exp(-df["Prediction"]))
        df["gradient"] = df["Raw"] - df["Passed"]
        df["Hessian"] = df["Raw"] * (1 - df["Raw"])

        trees.append({
            "threshold": threshold,
            "left_weight": left_weight,
            "right_weight": right_weight,
            "gain": gains[best_split]
        })

    return df, trees


In [46]:
result, trees = train_xgboost(df, n_trees=300)

print(result[["Hours", "Passed", "Prediction", "Raw", "gradient", "Hessian"]])
print(trees)

   Hours  Passed  Prediction       Raw  gradient   Hessian
0      1       0   -3.746263  0.023061  0.023061  0.022530
1      2       0   -3.746263  0.023061  0.023061  0.022530
2      3       0    0.001132  0.500283  0.500283  0.250000
3      4       1    0.001132  0.500283 -0.499717  0.250000
4      5       1    3.752730  0.977084 -0.022916  0.022391
5      6       1    3.752730  0.977084 -0.022916  0.022391
[{'threshold': np.float64(2.667), 'left_weight': np.float64(-0.6666666666666666), 'right_weight': np.float64(0.5), 'gain': np.float64(0.5833333333333333)}, {'threshold': np.float64(4.333), 'left_weight': np.float64(-0.4960520660029477), 'right_weight': np.float64(0.6501388605346471), 'gain': np.float64(0.5628497111254014)}, {'threshold': np.float64(4.333), 'left_weight': np.float64(-0.47146090176907746), 'right_weight': np.float64(0.629062831767298), 'gain': np.float64(0.5185515426258775)}, {'threshold': np.float64(2.667), 'left_weight': np.float64(-0.6136745635577777), 'right_wei

In [47]:
predicted_class = (result["Raw"] >= 0.5).astype(int)

accuracy = (predicted_class == result["Passed"]).mean()

print("Training accuracy:", accuracy)

Training accuracy: 0.8333333333333334
